# 🎙️ Gerador de Narração para Vídeo de Imóveis

Gera áudio de alta qualidade em **MP3** com voz masculina ou feminina para narração de vídeos de imóveis.  
Usa **Edge TTS** (Microsoft Neural Voices) — 100% gratuito, sem API key, vozes neurais de alta qualidade.

## Vozes disponíveis (Português Brasil):
| Gênero | Voz | Estilo |
|--------|-----|--------|
| Feminina | `pt-BR-FranciscaNeural` | Natural, clara, profissional |
| Masculina | `pt-BR-AntonioNeural` | Grave, confiante, profissional |
| Feminina 2 | `pt-BR-LeticiaNeural` | Jovem, dinâmica |
| Masculino 2 | `pt-BR-HumbertoNeural` | Formal, institucional |

## 1. Instalar dependências

In [1]:
# Instala edge-tts (gratuito, vozes neurais Microsoft)
!pip install edge-tts pydub

# Nota: ffmpeg é necessário para o pydub converter áudio
# Windows: baixe em https://ffmpeg.org/download.html e adicione ao PATH
# ou instale via: choco install ffmpeg  (se tiver chocolatey)
print('Dependências instaladas!')

  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pydub-0.25.1-py2.py3-none-any.whl (32 kB)

   -------------------- ------------------- 1/2 [edge-tts]
   ---------------------------------------- 2/2 [edge-tts]

Dependências instaladas!


## 2. Configuração — escolha voz e escreva o texto

In [ ]:
import os
from pathlib import Path

# ============================================================
# CONFIGURAÇÕES — edite aqui
# ============================================================

# Escolha a voz:
# 'feminina'  → pt-BR-FranciscaNeural  (recomendada — clara e profissional)
# 'masculina' → pt-BR-AntonioNeural    (recomendado — grave e confiante)
# 'feminina2' → pt-BR-LeticiaNeural
# 'masculina2'→ pt-BR-HumbertoNeural
GENERO = 'feminina'

# Velocidade: '-10%' (mais lento) | '+0%' (normal) | '+10%' (mais rápido)
VELOCIDADE = '-5%'

# Tom: '-10Hz' (mais grave) | '+0Hz' (normal) | '+10Hz' (mais agudo)
TOM = '+0Hz'

# Nome do arquivo de saída (sem extensão)
NOME_ARQUIVO = 'narracao_imovel'

# Pasta de saída
PASTA_SAIDA = Path('../outputs/audio')

# ============================================================
# TEXTO DA NARRAÇÃO — escreva aqui o roteiro do vídeo
# ============================================================
TEXTO = """
Apresentamos a você uma oportunidade única de morar no bairro mais valorizado da cidade.

Este deslumbrante apartamento de três quartos, com suíte, oferece o que há de melhor em 
conforto e sofisticação. São cento e vinte metros quadrados de puro requinte, com acabamento 
de alto padrão em todos os ambientes.

A sala integrada com varanda gourmet proporciona momentos inesquecíveis com a família e amigos. 
A cozinha planejada e os dois banheiros modernos completam este lar dos sonhos.

O condomínio conta com piscina, academia, salão de festas e segurança vinte e quatro horas.

Não perca esta chance. Entre em contato agora e agende sua visita!
"""

# Mapeamento de vozes
VOZES = {
    'feminina':  'pt-BR-FranciscaNeural',
    'masculina': 'pt-BR-AntonioNeural',
    'feminina2': 'pt-BR-LeticiaNeural',
    'masculina2':'pt-BR-HumbertoNeural',
}

voz_selecionada = VOZES.get(GENERO, 'pt-BR-FranciscaNeural')
print(f'Voz selecionada: {voz_selecionada}')
print(f'Velocidade: {VELOCIDADE} | Tom: {TOM}')
print(f'Texto ({len(TEXTO.split())} palavras):\n{TEXTO[:200]}...')

## 3. Gerar o áudio

In [ ]:
import asyncio
import edge_tts

# Cria a pasta de saída
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

# Caminho do arquivo final
arquivo_mp3 = PASTA_SAIDA / f'{NOME_ARQUIVO}_{GENERO}.mp3'
arquivo_srt = PASTA_SAIDA / f'{NOME_ARQUIVO}_{GENERO}.srt'  # legenda sincronizada

async def gerar_audio():
    communicate = edge_tts.Communicate(
        text=TEXTO.strip(),
        voice=voz_selecionada,
        rate=VELOCIDADE,
        pitch=TOM,
    )
    
    # Salva MP3 e coleta marcações de tempo para legenda
    submaker = edge_tts.SubMaker()
    with open(arquivo_mp3, 'wb') as f:
        async for chunk in communicate.stream():
            if chunk['type'] == 'audio':
                f.write(chunk['data'])
            elif chunk['type'] == 'WordBoundary':
                submaker.feed(chunk)
    
    # Salva legenda SRT sincronizada com o áudio
    with open(arquivo_srt, 'w', encoding='utf-8') as f:
        f.write(submaker.get_srt())
    
    return arquivo_mp3, arquivo_srt

# Executa geração
print('Gerando áudio... aguarde.')
mp3_path, srt_path = await gerar_audio()

tamanho_kb = os.path.getsize(mp3_path) / 1024
print(f'\n✅ Áudio gerado com sucesso!')
print(f'   MP3: {mp3_path} ({tamanho_kb:.1f} KB)')
print(f'   SRT: {srt_path}')

## 4. Ouvir o áudio direto no notebook

In [ ]:
from IPython.display import Audio, display

print(f'Reproduzindo: {arquivo_mp3.name}')
display(Audio(str(arquivo_mp3), autoplay=False))

## 5. (Opcional) Ajustar volume e qualidade com pydub

In [ ]:
# Requer ffmpeg instalado no sistema
try:
    from pydub import AudioSegment
    
    VOLUME_DB = 3   # +3 dB de volume (ajuste conforme necessário: -6 a +6)
    
    audio = AudioSegment.from_mp3(str(arquivo_mp3))
    audio_ajustado = audio + VOLUME_DB  # aumenta volume
    
    arquivo_final = PASTA_SAIDA / f'{NOME_ARQUIVO}_{GENERO}_final.mp3'
    audio_ajustado.export(
        str(arquivo_final),
        format='mp3',
        bitrate='192k',    # 192kbps — boa qualidade para vídeo
        tags={'artist': 'prompt2promolab', 'title': NOME_ARQUIVO}
    )
    
    print(f'✅ Áudio final exportado: {arquivo_final}')
    print(f'   Bitrate: 192kbps | Volume ajustado: +{VOLUME_DB}dB')
    print(f'   Duração: {len(audio_ajustado)/1000:.1f} segundos')
    display(Audio(str(arquivo_final), autoplay=False))

except ImportError:
    print('pydub não instalado ou ffmpeg não encontrado.')
    print('O arquivo MP3 original já está pronto em:', arquivo_mp3)
except Exception as e:
    print(f'Erro ao processar com pydub: {e}')
    print('Use o arquivo MP3 original:', arquivo_mp3)

## 6. Listar todas as vozes disponíveis em Português

In [ ]:
import asyncio
import edge_tts

async def listar_vozes_pt():
    vozes = await edge_tts.list_voices()
    pt_vozes = [v for v in vozes if v['Locale'].startswith('pt-BR')]
    print('Vozes disponíveis em Português (Brasil):\n')
    for v in pt_vozes:
        genero = '♀ Feminina' if v['Gender'] == 'Female' else '♂ Masculina'
        print(f"  {genero} | {v['ShortName']} | Estilos: {v.get('VoiceTag', {}).get('VoicePersonalities', ['N/A'])}")

await listar_vozes_pt()

## 7. Gerar versão masculina E feminina ao mesmo tempo

In [ ]:
import asyncio
import edge_tts
from IPython.display import Audio, display

async def gerar_par_vozes(texto, nome_base, pasta):
    configs = [
        ('feminina',  'pt-BR-FranciscaNeural'),
        ('masculina', 'pt-BR-AntonioNeural'),
    ]
    
    for genero, voz in configs:
        print(f'Gerando voz {genero} ({voz})...')
        communicate = edge_tts.Communicate(text=texto.strip(), voice=voz, rate='-5%')
        caminho = pasta / f'{nome_base}_{genero}.mp3'
        submaker = edge_tts.SubMaker()
        
        with open(caminho, 'wb') as f:
            async for chunk in communicate.stream():
                if chunk['type'] == 'audio':
                    f.write(chunk['data'])
                elif chunk['type'] == 'WordBoundary':
                    submaker.feed(chunk)
        
        srt_path = pasta / f'{nome_base}_{genero}.srt'
        with open(srt_path, 'w', encoding='utf-8') as f:
            f.write(submaker.get_srt())
        
        kb = os.path.getsize(caminho) / 1024
        print(f'  ✅ {caminho.name} ({kb:.1f} KB)')
        display(Audio(str(caminho), autoplay=False))

await gerar_par_vozes(TEXTO, NOME_ARQUIVO, PASTA_SAIDA)
print('\nAmbas as versões geradas! Escolha a que preferir.')

---
## Como usar no Camtasia Studio 8

1. **Importe o MP3**: No Camtasia, vá em `File > Import Media` e selecione o arquivo `.mp3` gerado
2. **Arraste para a timeline**: Coloque a faixa de áudio abaixo do vídeo na timeline
3. **Sincronize**: Alinhe o início do áudio com o início do vídeo
4. **Para usar o SRT gerado**: Veja o notebook `08_subtitle_generator.ipynb` para gerar legendas

### Arquivos gerados neste notebook:
```
outputs/audio/
├── narracao_imovel_feminina.mp3    ← áudio voz feminina
├── narracao_imovel_feminina.srt    ← legenda sincronizada com o áudio
├── narracao_imovel_masculina.mp3   ← áudio voz masculina  
└── narracao_imovel_masculina.srt   ← legenda sincronizada
```